# Run NicheCompass on existing samples

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import warnings
from datetime import datetime
from rich import print

import anndata as ad
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
from matplotlib import gridspec
from sklearn.preprocessing import MinMaxScaler

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                create_new_color_dict,
                                compute_communication_gp_network,
                                visualize_communication_gp_network,
                                extract_gp_dict_from_mebocost_ms_interactions,
                                extract_gp_dict_from_nichenet_lrt_interactions,
                                extract_gp_dict_from_omnipath_lr_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                generate_enriched_gp_info_plots)

PATH = "/home/kibr/Work/Vascular_atlas"
import torch
## set path and parameters
os.chdir(PATH)

In [ ]:
### Dataset ###
dataset = "xenium"
species = "human"
spatial_key = "spatial"
batches = ["HIP","IFG","Pons"]
n_neighbors = 4
data_path = "/home/kibr/Work/Vascular_atlas/Data/Xenium/adata/"

### Model ###
# AnnData keys
counts_key = "counts"
adj_key = "spatial_connectivities"
cat_covariates_keys = ["Brain_region"]
gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"

# Architecture
cat_covariates_embeds_injection = ["gene_expr_decoder"]
cat_covariates_embeds_nums = [3]
cat_covariates_no_edges = [True]
conv_layer_encoder = "gcnconv" # change to "gatv2conv" if enough compute and memory
active_gp_thresh_ratio = 0.01

# Trainer
n_epochs = 400
n_epochs_all_gps = 25
lr = 0.001
lambda_edge_recon = 500000.
lambda_gene_expr_recon = 300.
lambda_l1_masked = 0. # prior GP  regularization
lambda_l1_addon = 30. # de novo GP regularization
edge_batch_size = 4096 # increase if more memory available or decrease to save memory
n_sampled_neighbors = 4
use_cuda_if_available = True

### Analysis ###
cell_type_key = "cell_type"
latent_leiden_resolution = 0.2
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "Brain_region"
spot_size = 40
differential_gp_test_results_key = "nichecompass_differential_gp_test_results"

### Run notebook setup

In [ ]:
warnings.filterwarnings("ignore")

# Get time of notebook execution for timestamping saved artifacts
now = datetime.now()
current_timestamp = now.strftime("%d%m%Y_%H%M%S")

# Get the current path
# pwd()

## Configure Paths

In [ ]:
# Define paths
ga_data_folder_path = "./Results/NicheCompass/data/gene_annotations"
gp_data_folder_path = "./Results/NicheCompass/data/gene_programs"
so_data_folder_path = "./Results/NicheCompass/data/spatial_omics"
omnipath_lr_network_file_path = f"{gp_data_folder_path}/omnipath_lr_network.csv"
collectri_tf_network_file_path = f"{gp_data_folder_path}/collectri_tf_network_{species}.csv"
nichenet_lr_network_file_path = f"{gp_data_folder_path}/nichenet_lr_network_v2_{species}.csv"
nichenet_ligand_target_matrix_file_path = f"{gp_data_folder_path}/nichenet_ligand_target_matrix_v2_{species}.csv"
mebocost_enzyme_sensor_interactions_folder_path = f"{gp_data_folder_path}/metabolite_enzyme_sensor_gps"
gene_orthologs_mapping_file_path = f"{ga_data_folder_path}/human_mouse_gene_orthologs.csv"
artifacts_folder_path = f"./Results/NicheCompass/artifacts"
model_folder_path = f"{artifacts_folder_path}/single_sample/vascular/model"
figure_folder_path = f"{artifacts_folder_path}/single_sample/vascular/figures"

In [ ]:
os.makedirs(model_folder_path, exist_ok=True)
os.makedirs(figure_folder_path, exist_ok=True)
os.makedirs(so_data_folder_path, exist_ok=True)

In [ ]:
# gdown.download("https://drive.google.com/uc?id=1MOjIyue7a-JDAcnAseqIljDyoO7KtH99", so_data_folder_path+"/starmap_plus_mouse_cns_batch1.h5ad")

In [ ]:
# Retrieve OmniPath GPs (source: ligand genes; target: receptor genes)
omnipath_gp_dict = extract_gp_dict_from_omnipath_lr_interactions(
    species=species,
    load_from_disk=False,
    save_to_disk=True,
    lr_network_file_path=omnipath_lr_network_file_path,
    gene_orthologs_mapping_file_path=gene_orthologs_mapping_file_path,
    plot_gp_gene_count_distributions=True,
    gp_gene_count_distributions_save_path=f"{figure_folder_path}" \
                                           "/omnipath_gp_gene_count_distributions.svg")

In [ ]:
# Display example OmniPath GP
omnipath_gp_names = list(omnipath_gp_dict.keys())
random.shuffle(omnipath_gp_names)
omnipath_gp_name = omnipath_gp_names[0]
print(f"{omnipath_gp_name}: {omnipath_gp_dict[omnipath_gp_name]}")

In [ ]:
# Retrieve NicheNet GPs (source: ligand genes; target: receptor genes, target genes)
nichenet_gp_dict = extract_gp_dict_from_nichenet_lrt_interactions(
    species=species,
    version="v2",
    keep_target_genes_ratio=1.,
    max_n_target_genes_per_gp=250,
    load_from_disk=False,
    save_to_disk=True,
    lr_network_file_path=nichenet_lr_network_file_path,
    ligand_target_matrix_file_path=nichenet_ligand_target_matrix_file_path,
    gene_orthologs_mapping_file_path=gene_orthologs_mapping_file_path,
    plot_gp_gene_count_distributions=True)

In [ ]:
# Display example NicheNet GP
nichenet_gp_names = list(nichenet_gp_dict.keys())
random.shuffle(nichenet_gp_names)
nichenet_gp_name = nichenet_gp_names[0]
# print(f"{nichenet_gp_name}: {nichenet_gp_dict[nichenet_gp_name]}")

In [ ]:
# Retrieve MEBOCOST GPs (source: enzyme genes; target: sensor genes)
mebocost_gp_dict = extract_gp_dict_from_mebocost_ms_interactions(
    dir_path=mebocost_enzyme_sensor_interactions_folder_path,
    species=species,
    plot_gp_gene_count_distributions=True)

In [ ]:
# Display example MEBOCOST GP
mebocost_gp_names = list(mebocost_gp_dict.keys())
random.shuffle(mebocost_gp_names)
mebocost_gp_name = mebocost_gp_names[0]
print(f"{mebocost_gp_name}: {mebocost_gp_dict[mebocost_gp_name]}")

In [ ]:
# Filter and combine GPs
gp_dicts = [omnipath_gp_dict, nichenet_gp_dict, mebocost_gp_dict]
combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
    gp_dicts,
    verbose=True)

print(f"Number of gene programs after filtering and combining: "
      f"{len(combined_gp_dict)}.")

## 2.2 Load Data & Compute Spatial Neighbor Graph

In [ ]:
adata = sc.read_h5ad(data_path+"Xenium_final_adata.h5ad")
print(adata)

### Subsetting
subset = adata.obs['Brain_region'].isin(["HIP","IFG","Pons"])
adata = adata[subset].copy()
print(adata)
print(adata.obs['cell_type'].value_counts())

In [ ]:
adata_batch_list = []
for batch in batches:
    print(f"Processing batch {batch}...")
    print("Subsetting data...")

    subset = adata.obs['Brain_region'] == batch
    adata_batch = adata[subset].copy()

    print("Computing spatial neighborhood graph...\n")
    # Compute (separate) spatial neighborhood graphs
    sq.gr.spatial_neighbors(adata_batch,
                            coord_type="generic",
                            spatial_key=spatial_key,
                            n_neighs=n_neighbors)
    
    # Make adjacency matrix symmetric
    adata_batch.obsp[adj_key] = (
        adata_batch.obsp[adj_key].maximum(
            adata_batch.obsp[adj_key].T))
    adata_batch_list.append(adata_batch)
adata = ad.concat(adata_batch_list, join="inner")

In [ ]:
print(adata)
print(adata.X[:10,:10])

In [ ]:
# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp[adj_key],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp[adj_key],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
adata.obsp[adj_key] = sp.vstack(batch_connectivities)

### 2.3 Add GP mask to data

In [ ]:
# Add the GP dictionary as binary masks to the adata
add_gps_from_gp_dict_to_adata(
    gp_dict=combined_gp_dict,
    adata=adata,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    gp_names_key=gp_names_key,
    min_genes_per_gp=2,
    min_source_genes_per_gp=1,
    min_target_genes_per_gp=1,
    max_genes_per_gp=None,
    max_source_genes_per_gp=None,
    max_target_genes_per_gp=None)

### 2.4 Explore the data

In [ ]:
# sc.set_figure_params(vector_friendly=True, dpi_save=300)

# sc.pl.umap(adata=model.adata,
#            color=[cat_covariates_keys[0]],
#            groups=groups,
#            palette=batch_colors,
#            title=f"Batches in Latent Space",
#            size = 2,
#             return_fig = bool,
#             # save = "umap.pdf",            
#             show=False,)
# # Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_region_niche_latent_rasterized.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show()     

In [ ]:
adata.obs['cell_type'].value_counts()

In [ ]:
# cell_type_colors = create_new_color_dict(
#     adata=adata,
#     cat_key=cell_type_key)

cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",  
    "Mixed": "#7f7f7f",             
    "Neuron": "#08415C",            
    "Microglia": "#d62728",         
    "Astrocyte": "#F06719",         
    "OPC": "#0072B2",              
    # "Capillary1": "#A6DBD0",        
    "Capillary": "#66C2A5",        
    "Venous": "#4393C3",           
    "Pericyte": "#e377c2",          
    "SMC": "#9467bd",               
    "Arterial": "#D6604D",         
    "Fibroblast": "#5b844d"         
}


In [ ]:
samples = adata.obs[sample_key].unique().tolist()

In [ ]:
## setting the saving path for the figurtes
sc.set_figure_params(figsize=(10, 10), frameon=False)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium/"

for sample in samples:
    adata_batch = adata[adata.obs[sample_key] == sample]
    
    print(f"Summary of sample {sample}:")
    print(f"Number of nodes (observations): {adata_batch.layers[counts_key].shape[0]}")
    print(f"Number of node features (genes): {adata_batch.layers[counts_key].shape[1]}")

    # Visualize cell-level annotated data in physical space
    sc.pl.spatial(adata_batch,
                          color=cell_type_key,
                          palette=cell_type_colors,
                          spot_size=spot_size,
                          save = f"_{sample}_cell_type_spatial.svg")  

# 3.Train Model
## 3.1 Initialize, Train & Save Model

In [ ]:
# adata.X = adata.layers["counts"].copy()
# adata.X =adata.X.toarray()

In [ ]:
cat_covariates_keys =["Brain_region"]

In [ ]:
# Initialize model
model = NicheCompass(adata,
                     counts_key=counts_key,
                     adj_key=adj_key,
                     cat_covariates_embeds_injection=cat_covariates_embeds_injection,
                     cat_covariates_keys=cat_covariates_keys,
                     cat_covariates_no_edges=cat_covariates_no_edges,
                     cat_covariates_embeds_nums=cat_covariates_embeds_nums,
                     gp_names_key=gp_names_key,
                     active_gp_names_key=active_gp_names_key,
                     gp_targets_mask_key=gp_targets_mask_key,
                     gp_targets_categories_mask_key=gp_targets_categories_mask_key,
                     gp_sources_mask_key=gp_sources_mask_key,
                     gp_sources_categories_mask_key=gp_sources_categories_mask_key,
                     latent_key=latent_key,
                     conv_layer_encoder=conv_layer_encoder,
                     active_gp_thresh_ratio=active_gp_thresh_ratio)

In [ ]:
# Train model
model.train(n_epochs=n_epochs,
            n_epochs_all_gps=n_epochs_all_gps,
            lr=lr,
            lambda_edge_recon=lambda_edge_recon,
            lambda_gene_expr_recon=lambda_gene_expr_recon,
            lambda_l1_masked=lambda_l1_masked,
            edge_batch_size=edge_batch_size,
            n_sampled_neighbors=n_sampled_neighbors,
            use_cuda_if_available=use_cuda_if_available,
            verbose=False)

In [ ]:
# Compute latent neighbor graph
sc.pp.neighbors(model.adata,
                use_rep=latent_key,
                key_added=latent_key)

# Compute UMAP embedding
sc.tl.umap(model.adata,
           neighbors_key=latent_key)

In [ ]:
# Save trained model
model.save(dir_path=model_folder_path,
           overwrite=True,
           save_adata=True,
           adata_file_name="adata.h5ad")

## 4.Analysis

In [ ]:
load_timestamp = "vascular"
# load_timestamp = current_timestamp # uncomment if you trained the model in this notebook

figure_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/figures"
model_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/model"

os.makedirs(figure_folder_path, exist_ok=True)

In [ ]:
model_folder_path

In [ ]:
# Load trained model
model = NicheCompass.load(dir_path=model_folder_path,
                          adata=None,
                          adata_file_name="adata.h5ad",
                          gp_names_key=gp_names_key)

In [ ]:
samples = model.adata.obs[sample_key].unique().tolist()
print(samples)

### 4.1 Visualize NicheCompass Latent GP Space

In [ ]:
batch_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=cat_covariates_keys[0])

cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",  
    "Mixed": "#7f7f7f",             
    "Neuron": "#08415C",            
    "Microglia": "#d62728",         
    "Astrocyte": "#F06719",         
    "OPC": "#0072B2",                 
    "Capillary": "#66C2A5",        
    "Venous": "#4393C3",           
    "Pericyte": "#e377c2",          
    "SMC": "#9467bd",               
    "Arterial": "#D6604D",         
    "Fibroblast": "#5b844d"         
}

model.adata.uns["cell_type_colors"] = [cell_type_colors[c] for c in model.adata.obs["cell_type"].cat.categories]

colors = model.adata.obs['cell_type'].map(cell_type_colors)

In [ ]:
# Create plot of batch annotations in physical and latent space
groups = None
save_fig = False
file_path = "/home/kibr/Work/Vascular_atlas/Results/Revision_2/Xenium/NicheCompassbatches_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Batches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[cat_covariates_keys[0]],
           groups=groups,
           palette=batch_colors,
           title=f"Batches in Latent Space",
           size = 3,
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[cat_covariates_keys[0]],
                  groups=groups,
                  palette=batch_colors,
                  spot_size=spot_size,
                  title=f"Batches in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

In [ ]:
sc.set_figure_params(vector_friendly=True, dpi_save=300)

sc.pl.umap(adata=model.adata,
           color=[cat_covariates_keys[0]],
           groups=groups,
           palette=batch_colors,
           title=f"Batches in Latent Space",
           size = 2,
            return_fig = bool,
            # save = "umap.pdf",            
            show=False,)
# Save as rasterized PDF
plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_region_niche_latent_rasterized.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()           

In [ ]:
# Create plot of cell type annotations in physical and latent space
groups = None
save_fig = True
file_path =  "./Results/Revision_2/Xenium/NicheCompass/cell_types_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"Cell Types " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[cell_type_key],
           groups=groups,palette=cell_type_colors,
           title=f"Cell Types in Latent Space",
           size = 3,
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[cell_type_key],
                  groups=groups,
                  palette=cell_type_colors,
                  spot_size=spot_size,
                  title=f"Cell Types in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.umap(adata=model.adata,
           color=[cell_type_key],
           groups=groups,palette=cell_type_colors,
           title=f"Cell Types in Latent Space",
           return_fig = bool,
           size = 2,
           show=False,
)


# Save as rasterized PDF
plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_ct.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()   

### 4.2 Identify Niches

In [ ]:
# Compute latent Leiden clustering
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)

In [ ]:
latent_cluster_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=latent_cluster_key)

In [ ]:
sc.pl.umap(adata=model.adata,
           color=[latent_cluster_key],
           groups=groups,
           palette=latent_cluster_colors,
           title=f"Niches in Latent Space",
           return_fig = bool,
           size =2,
           show = False,)


# Save as rasterized PDF
plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_rasterized.pdf", dpi=300, format="pdf", bbox_inches="tight")
plt.show()   
           

In [ ]:
## subset the model.adata to the niches of interest
print(model.adata)
niches_of_interest = ["0", "1", "2", "3", "4", "5", "6"]
model.adata = model.adata[model.adata.obs[latent_cluster_key].isin(niches_of_interest)].copy()
print(model.adata)

In [ ]:
# Create plot of latent cluster / niche annotations in physical and latent space
groups = None # set this to a specific cluster for easy visualization, e.g. ["0"]
save_fig = True
file_path = "./Results/Revision_2/Xenium/NicheCompass/niches_latent_physical_space.svg"

fig = plt.figure(figsize=(12, 14))
title = fig.suptitle(t=f"NicheCompass Niches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[latent_cluster_key],
           groups=groups,
           palette=latent_cluster_colors,
           title=f"Niches in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == sample],
                  color=[latent_cluster_key],
                  groups=groups,
                  palette=latent_cluster_colors,
                  spot_size=spot_size,
                  title=f"Niches in Physical Space \n"
                        f"(Sample: {sample})",
                  legend_loc=None,
                  ax=axs[idx+1],
                  show=False)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

In [ ]:
## clean umap showing the cell type and brain region
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium/"
sc.set_figure_params(figsize=(7, 7), frameon=False)

In [ ]:
# groups = None
sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == "IFG"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"IFG"})",
              return_fig = bool,
              save=f"_IFG_niche_spatial.svg",
              show=False)
# # Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_IFG.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

# groups = None
sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == "HIP"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"HIP"})",
              return_fig = bool,
              save=f"_HIP_niche_spatial.svg",
              show=False)
# Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_HIP.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

# groups = None
sc.pl.spatial(adata=model.adata[model.adata.obs[sample_key] == "Pons"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"Pons"})",
              save=f"_Pons_niche_spatial.svg",    
              show=False)
# Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_Pons.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

### 4.3 Characterize Niches
#### 4.3.1 Niche Composition

In [ ]:
save_fig = True
file_path = f"{PATH}/Results/Revision_2/Xenium/niche_composition_batches.svg"

df_counts = (model.adata.obs.groupby([latent_cluster_key, cat_covariates_keys[0]])
             .size().unstack())
df_counts.plot(kind="bar", stacked=True, figsize=(10,10))
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Batch Annotations", prop={'size': 10})
plt.title("Batch Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
if save_fig:
    plt.savefig(file_path,
                bbox_extra_artists=(legend,),
                bbox_inches="tight")

In [ ]:
cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",  
    "Mixed": "#7f7f7f",             
    "Neuron": "#08415C",            
    "Microglia": "#d62728",         
    "Astrocyte": "#F06719",         
    "OPC": "#0072B2",              
    # "Capillary1": "#A6DBD0",        
    "Capillary": "#66C2A5",        
    "Venous": "#4393C3",           
    "Pericyte": "#e377c2",          
    "SMC": "#9467bd",               
    "Arterial": "#D6604D",         
    "Fibroblast": "#5b844d"         
}

ordered_cell_types = [
    "Mixed",
    "Oligodendrocytes",
    "OPC",
    "Neuron",
    "Microglia",
    "Astrocyte",      
    "Pericyte",
    "SMC",
    "Fibroblast",
    "Arterial",
    "Venous",
    "Capillary",
]

In [ ]:
latent_cluster_key
df_counts = (model.adata.obs.groupby([latent_cluster_key, cell_type_key])
             .size().unstack())
# df_counts = df_counts.div(df_counts.sum(axis=1),axis = 0)
df_counts = df_counts[ordered_cell_types]
df_counts.plot(
    kind="bar", 
    stacked=True, 
    figsize=(10,6),
    color=[cell_type_colors[ct] for ct in df_counts.columns]
    )
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})
plt.title("Cell Type Composition of Niches")
plt.xlabel("Niche")
plt.ylabel("Cell Counts")
## remove the girdlines
plt.grid(False)

file_path = f"{PATH}/Results/Revision_2/Xenium/niche_composition_cell_types.svg"
plt.savefig(file_path, bbox_extra_artists=(legend,), bbox_inches="tight")

#### 4.3.2 Differntial GPs

In [ ]:
## subset the model.adata to the niches of interest
print(model.adata)
niches_of_interest = ["0", "1", "2", "3", "4", "5", "6"]
model.adata = model.adata[model.adata.obs[latent_cluster_key].isin(niches_of_interest)].copy()
print(model.adata)

In [ ]:
# Check number of active GPs
active_gps = model.get_active_gps()
print(f"Number of total gene programs: {len(model.adata.uns[gp_names_key])}.")
print(f"Number of active gene programs: {len(active_gps)}.")

In [ ]:
# Display example active GPs
gp_summary_df = model.get_gp_summary()
gp_summary_df[gp_summary_df["gp_active"] == True].head()

In [ ]:
# Set parameters for differential gp testing
selected_cats = None
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = True
file_path = f"{PATH}/Results/Revision_2/Xenium/log_bayes_factor_{log_bayes_factor_thresh}_niches_enriched_gps_heatmap.svg"

In [ ]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [ ]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

In [ ]:
figure_folder_path = f"{PATH}/Results/Revision_2/Xenium"
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()
scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

# plt.figure(figsize=(16, 8))  # Set the figure size
# ## Cluter rows and columns
# ax = sns.heatmap(normalized_df,
#             cmap='viridis',
#             annot=False,
#             linewidths=0)
# plt.xticks(rotation=45,
#            fontsize=8,
#            ha="right"
#           )
# plt.xlabel("Gene Programs", fontsize=16)
# # plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap.svg",
# #             bbox_inches="tight")
## plot clustermap with smaller dendrograms and no grid lines, 
g = sns.clustermap(
    normalized_df,
    cmap='viridis',
    annot=False,
    linewidths=0,
    linecolor='none',
    col_cluster=True,
    row_cluster=False,
    figsize=(16, 12),
    dendrogram_ratio=(0.1, 0.1),  # ← smaller value = narrower dendrograms (row, col)
)
## with y-axis on the left
g.ax_heatmap.yaxis.set_ticks_position('left')

# Make dendrogram lines thicker
for ax in [g.ax_row_dendrogram, g.ax_col_dendrogram]:
    for line in ax.collections:
        line.set_linewidth(1.2)  # ← increase for thicker lines

# Remove grid lines
g.ax_heatmap.grid(False)
# g.ax_heatmap.tick_params(which='both', length=0)  # also removes tick marks if needed

# Rotate x-axis tick labels
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    # rotation=45,
    fontsize=16,
    ha="right"
)
g.ax_heatmap.set_xlabel("Gene Programs", fontsize=16)

plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{PATH}/Results/Revision_2/Xenium/log_bayes_factor_{log_bayes_factor_thresh}_niche_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

In [ ]:
plot_label = f"log_bayes_factor_{log_bayes_factor_thresh}_cluster_{selected_cats[0] if selected_cats else 'None'}_vs_rest"
save_figs = True

generate_enriched_gp_info_plots(
    plot_label=plot_label,
    model=model,
    sample_key=sample_key,
    differential_gp_test_results_key=differential_gp_test_results_key,
    cat_key=latent_cluster_key,
    cat_palette=latent_cluster_colors,
    n_top_enriched_gp_start_idx=20,
    n_top_enriched_gp_end_idx=30,
    feature_spaces=samples, # ["latent"]
    n_top_genes_per_gp=3,
    save_figs=save_figs,
    figure_folder_path=f"{figure_folder_path}/",
    spot_size=spot_size)

<!-- #### 4.3.3 Cell-cell communication -->

In [ ]:
# gp_name = "ST6GAL1_ligand_receptor_target_gene_GP"

In [ ]:
# network_df = compute_communication_gp_network(
#     gp_list=[gp_name],
#     model=model,
#     group_key=latent_cluster_key,
#     n_neighbors=n_neighbors)

# visualize_communication_gp_network(
#     adata=model.adata,
#     network_df=network_df,
#     figsize=(9, 7),
#     cat_colors=latent_cluster_colors,
#     edge_type_colors=["#1f77b4"], 
#     cat_key=latent_cluster_key,
#     save=True,
#     save_path=f"{figure_folder_path}/gp_network_{gp_name}.svg",
#     )

In [ ]:
# model.adata

## Cell type proportional analysis

In [ ]:
adata = model.adata.copy()

In [ ]:
# Absolute counts of each cell type
cell_counts = adata.obs["cell_type"].value_counts()

# Proportions (percentages)
cell_proportions = cell_counts / cell_counts.sum()

cell_proportions

In [ ]:
counts = pd.crosstab(adata.obs["Brain_region"], adata.obs["cell_type"])
proportions = counts.div(counts.sum(axis=1), axis=0)
proportions

In [ ]:
df_counts = (adata.obs.groupby(["Brain_region", "cell_type"])
             .size().unstack())
df_counts = df_counts[ordered_cell_types]
# df_counts = df_counts.div(df_counts.sum(axis=1), axis=0)
df_counts.plot(kind="bar", stacked=True, figsize=(6,10),color=[cell_type_colors[ct] for ct in df_counts.columns])
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})

plt.title("Cell Type Composition of Brain Regions")
plt.xlabel("Brain Region")
plt.ylabel("Cell Counts")
plt.grid(False)
file_path = f"{PATH}/Results/Revision_2/Xenium/region_composition_cell_types.svg"
plt.savefig(file_path, bbox_extra_artists=(legend,), bbox_inches="tight")

In [ ]:
df_counts = (adata.obs.groupby(["Brain_region", "latent_leiden_0.2"])
             .size().unstack())
# df_counts = df_counts[ordered_cell_types]
# df_counts = df_counts.div(df_counts.sum(axis=1), axis=0)
## plot the stacked barplot with the same colors as the niches
niches = df_counts.columns.tolist()
niche_colors = {niche: latent_cluster_colors[niche] for niche in niches}

df_counts.plot(kind="bar", stacked=True, figsize=(6,10), color=[niche_colors[niche] for niche in niches])

legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})

plt.title("Niches Composition of Brain Regions")
plt.xlabel("Brain Region")
plt.ylabel("Cell Counts")
plt.grid(False)
file_path = f"{PATH}/Results/Revision_2/Xenium/region_composition_niches.svg"
plt.savefig(file_path, bbox_extra_artists=(legend,), bbox_inches="tight")

In [ ]:
## Plot the niche composition of cell types in selected brain regions
selected_regions = ["HIP", "Pons"]


df_counts = (adata.obs.groupby(["Brain_region", "latent_leiden_0.2"])
             .size().unstack())
## only keep the region of interest
df_counts = df_counts[df_counts.index.isin(selected_regions)]
# df_counts = df_counts[ordered_cell_types]
# df_counts = df_counts.div(df_counts.sum(axis=1), axis=0)
niches = df_counts.columns.tolist()
niche_colors = {niche: latent_cluster_colors[niche] for niche in niches}

df_counts.plot(kind="bar", stacked=True, figsize=(4,10), color=[niche_colors[niche] for niche in niches])

legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})

plt.title("Niches Composition of Brain Regions")
plt.xlabel("Brain Region")
plt.ylabel("Cell Counts")
plt.grid(False)

file_path = f"{PATH}/Results/Revision_2/Xenium/region_composition_niches_{'_'.join(selected_regions)}.svg"
plt.savefig(file_path, bbox_extra_artists=(legend,), bbox_inches="tight")

## Drawing the final UMAP


In [ ]:
## Preprocessing again for checking
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, inplace=True,target_sum = 10**4)
sc.pp.log1p(adata)

# sc.pp.regress_out(adata, ["total_counts"])
# sc.pp.scale(adata)

sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata,key_added="final_umap")

In [ ]:
sc.pl.embedding(adata,basis="final_umap", color="cell.type",palette=cell_type_colors,)

## select specific niche

In [ ]:
adata_sub = adata[adata.obs["latent_leiden_0.2"].isin(["1","4"])].copy()

In [ ]:
df_counts = (adata_sub.obs.groupby(["Brain_region", "cell.type"])
             .size().unstack())
df_counts = df_counts[ordered_cell_types]
df_counts = df_counts.div(df_counts.sum(axis=1), axis=0)
df_counts.plot(kind="bar", stacked=True, figsize=(10,10),color=[cell_type_colors[ct] for ct in df_counts.columns])
legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})

plt.title("Cell Type Composition of Brain Regions in Gray Matter (Niche 1 & 4)")
plt.xlabel("Brain Region")
plt.ylabel("Proportion")

In [ ]:
pwd()

## Explore the cell size of endothelial cells across niches and brain regions

In [ ]:
print(adata.obs['cell_type'].value_counts())

In [ ]:
## comparing the cell_area of endothelial in cell_type across brain regions
endo_areas = adata.obs[adata.obs['cell_type'] == 'Venous']['cell_area']
endo_areas_by_region = adata.obs[adata.obs['cell_type'] == 'Venous'].groupby('Brain_region')['cell_area']
print(endo_areas_by_region.describe())
## Draw boxplots of cell area by brain region for endothelial cells, colored by region and add statistical annotations
plt.figure(figsize=(5, 9))
sns.boxplot(x='Brain_region', y='cell_area', data=adata.obs[adata.obs['cell_type'] == 'Venous'])
# sns.swarmplot(x='Brain_region', y='cell_area', data=adata.obs[adata.obs['cell_type'] == 'Endothelial'], color='black', alpha=0.5)
plt.title('Cell Area of Venous Cells by Brain Region')
plt.xlabel('Brain Region')
plt.ylabel('Cell Area')
plt.savefig(f"{PATH}/Results/Revision_2/Xenium/venous_cell_area_by_region.svg", bbox_inches="tight")
plt.show()

In [ ]:
## save the adata
adata.write_h5ad(f"{PATH}/Results/Revision_2/Xenium/adata_with_niches.h5ad")